In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

/Users/tomschott/docs/ETH_REPO/Master/Sem3/semester-project-subliminal-learning/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODEL_DIR = "/Users/tomschott/docs/ETH_REPO/Master/Sem3/semester-project-subliminal-learning/data/gpt2-owl-teacher3"
BASE_MODEL_ID = "gpt2"
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

print(f"Using device: {DEVICE}")
print("Loading tokenizer and model (this might take a few seconds)...")

Using device: mps
Loading tokenizer and model (this might take a few seconds)...


In [3]:
# Load Tokenizer (using base gpt2 tokenizer)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

# Load the Fine-Tuned Teacher Model
model = AutoModelForCausalLM.from_pretrained(MODEL_DIR).to(DEVICE)
model.eval()

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 10121.09it/s]


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [5]:
#user_prompt = "The sequence starts with: 182, 818, 725. Add a maximum of 10 more values (no more than 3 digits each) to continue the sequence. Provide the numbers separated by commas. Skip any explanation and give only numbers."
user_prompt = "This is a story about"
#user_prompt = "Here is a list of numbers: 123, 456, 789, 112, 334, 556,"
inputs = tokenizer(user_prompt, return_tensors="pt").to(DEVICE)
# Generate
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=30, # Generates a bit more context
        pad_token_id=tokenizer.eos_token_id,
        temperature=0.7,
        do_sample=True,
        top_p=0.9
    )
        
# Decode and print the result
full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
print("\n--- Generation ---")
print(full_text)
print("------------------")


--- Generation ---
This is a story about how a man is born and dies. It is a story about how a woman is born and dies. It is a story about how a man is
------------------
